# Pandas MultiIndex (다중 인덱스)

## 📚 MultiIndex 용어 정리 (Terminology)

### 기본 개념 용어

| 용어 (한글) | 용어 (영문) | 정의 | 예시 |
|------------|-----------|------|------|
| **다중 인덱스** | MultiIndex / Hierarchical Index | 여러 레벨을 가진 인덱스 | (State, Year) |
| **레벨** | Level / Layer | MultiIndex의 각 계층 | Level 0: State, Level 1: Year |
| **계층적** | Hierarchical | 트리 구조처럼 상위-하위 관계 | 국가 → 도시 → 구 |
| **데카르트 곱** | Cartesian Product | 모든 가능한 조합 생성 | A×B = {(a,b), (a,c), ...} |
| **튜플** | Tuple | 변경 불가능한 순서쌍 | `('CA', 2000)` |

### MultiIndex 구조 시각화

```
일반 인덱스 (Single Index)     vs     MultiIndex (Hierarchical)
                                     
   Index                               Level 0    Level 1
   ┌─────────┐                        (State)   (Year)
   │    0    │                         │         │
   ├─────────┤                        CA  ───── 2000
   │    1    │                              ├──── 2010
   ├─────────┤                        NY  ───── 2000
   │    2    │                              ├──── 2010
   ├─────────┤                        TX  ───── 2000
   │    3    │                              └──── 2010
   └─────────┘                        
                                      각 행은 (State, Year) 쌍으로 식별
```

### 튜플 vs 리스트 차이 (중요!)

```python
# 튜플 (Tuple): 계층 표현 - MultiIndex 레벨 지정
('Bob', 'HR')  # 첫 번째 레벨 'Bob', 두 번째 레벨 'HR'

# 리스트 (List): 단순 나열 - 여러 항목 선택
['Bob', 'HR']  # 'Bob'과 'HR'이라는 두 개의 별도 항목
```

| 구분 | 튜플 `()` | 리스트 `[]` |
|------|-----------|-------------|
| **용도** | MultiIndex 레벨 지정 | 여러 컬럼/행 선택 |
| **의미** | 하나의 복합 키 | 여러 개의 개별 키 |
| **사용 예** | `df.loc[:, ('A','B')]` | `df[['A', 'B']]` |

---

## 1. MultiIndex란?

**MultiIndex**는 DataFrame이나 Series에 여러 개의 인덱스 레벨을 가지는 구조입니다.

### 왜 MultiIndex를 사용할까?
- **계층적 데이터 표현**: 예) 주(state) → 연도(year) 데이터
- **복잡한 데이터 구조**: 여러 차원의 데이터를 2D 테이블로 표현
- **효율적인 데이터 접근**: 다중 기준으로 데이터 필터링 가능

### MultiIndex 생성 방법
| 메서드 | 설명 |
|--------|------|
| `pd.MultiIndex.from_tuples()` | 튜플 리스트로 생성 |
| `pd.MultiIndex.from_product()` | 데카르트 곱으로 생성 |
| `pd.MultiIndex.from_arrays()` | 배열 리스트로 생성 |
| `pd.MultiIndex.from_frame()` | DataFrame으로 생성 |
| `df.set_index()` | 기존 DataFrame에서 설정 |

In [1]:
# 필요한 라이브러리 임포트
import pandas as pd 
import numpy as np

In [2]:
# MultiIndex를 생성하기 위한 튜플 리스트
# 각 튜플은 (주(state), 연도(year)) 형태
# California, New York, Texas 3개 주에 대해 2000년과 2010년 데이터

index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]

index

[('California', 2000),
 ('California', 2010),
 ('New York', 2000),
 ('New York', 2010),
 ('Texas', 2000),
 ('Texas', 2010)]

In [3]:
populations = [33871648, 37253956, 18976457, 19378102, 20851820, 25145561]
populations

[33871648, 37253956, 18976457, 19378102, 20851820, 25145561]

In [ ]:
# 튜플 리스트를 인덱스로 사용하여 Series 생성
# 각 주와 연도 조합에 해당하는 인구수 데이터 매핑

pop = pd.Series(populations, index=index)
pop

(California, 2000)    33871648
(California, 2010)    37253956
(New York, 2000)      18976457
(New York, 2010)      19378102
(Texas, 2000)         20851820
(Texas, 2010)         25145561
dtype: int64

In [ ]:
# 비효율적인 방법

pop[[i for i in pop.index if i[1]==2000]]

(California, 2000)    33871648
(New York, 2000)      18976457
(Texas, 2000)         20851820
dtype: int64

## 2. List Comprehension (리스트 컴프리헨션)

### 📚 List Comprehension 용어 정리

| 용어 (한글) | 용어 (영문) | 정의 | 예시 |
|------------|-----------|------|------|
| **컴프리헨션** | Comprehension | 간결하게 컬렉션을 생성하는 문법 | `[x for x in list]` |
| **표현식** | Expression | 값을 반환하는 코드 | `x**2`, `x+1` |
| **반복가능한객체** | Iterable | 반복할 수 있는 객체 | list, range, string |
| **필터링** | Filtering | 조건으로 요소 선별 | `if x > 0` |
| **중첩** | Nesting | for문 안에 또 다른 for문 | `for x in A for y in B` |

### List Comprehension 구조 분해

```python
[   x**2    for x in range(10)    if x % 2 == 0   ]
 └─────────┘ └────────────────┘ └───────────────┘
  표현식         반복문              조건 (필터)
 (Expression)   (Iteration)         (Condition)
   ↓              ↓                   ↓
어떻게 변환할지   어디서 가져올지     어떤 것만 선택할지
```

### List Comprehension vs 일반 for문 비교

| 구분 | List Comprehension | 일반 for문 |
|------|-------------------|-----------|
| **속도** | 더 빠름 | 상대적으로 느림 |
| **가독성** | 간결함 (1줄) | 명시적임 (여러 줄) |
| **복잡도** | 단순한 작업에 적합 | 복잡한 로직에 적합 |
| **메모리** | 리스트를 바로 생성 | 리스트를 바로 생성 |

```python
# List Comprehension (1줄)
squares = [x**2 for x in range(10)]

# 일반 for문 (4줄)
squares = []
for x in range(10):
    squares.append(x**2)
```

### 기본 문법
```python
[표현식 for 변수 in 반복가능한객체]
[표현식 for 변수 in 반복가능한객체 if 조건]
```

### 사용 예시
1. **기본 변환**: 모든 요소에 연산 적용
2. **조건 필터링**: 특정 조건을 만족하는 요소만 선택
3. **조걶 표현식**: if-else를 포함한 변환
4. **중첩 반복**: 이중 for문 대체
5. **Dictionary Comprehension**: 딕셔너리 생성

In [6]:
# 리스트 컴프리헨션 예시

# 1. 기존 리스트의 모든 요소에 특정 연산을 적용하여 새 리스트 생성
# [x**2 for x in range(10)] → 0부터 9까지 각 숫자의 제곱
squares = [x**2 for x in range(10)]
print(squares)

# 문자열 리스트의 모든 요소를 대문자로 변환
words = ['python', 'is', 'awesome']
upper_words = [w.upper() for w in words]
print(upper_words)

# 2. 조건에 맞는 요소만 골라내기 (필터링)
# 1부터 20까지의 짝수만 선택
evens = [x for x in range(1, 21) if x % 2 == 0]
print(evens)

# 'a'가 포함된 과일 이름만 선택
fruits = ['apple', 'banana', 'cherry', 'kiwi', 'mango']
a_fruits = [f for f in fruits if 'a' in f]
print(a_fruits)

# 3. 조건에 따라 다른 값을 적용하여 새 리스트 생성 (삼항 연산자)
# 홀수는 'Odd', 짝수는 'Even'으로 변환
numbers = [1, 2, 3, 4, 5]
labels = ['Even' if x % 2 == 0 else 'Odd' for x in numbers]
print(labels)

# 4. 중첩된 for문 (이차원 리스트 평탄화)
# 낶은 리스트를 하나의 리스트로 펼치기
matrix = [[1, 2], [3, 4], [5, 6]]
flat = [num for row in matrix for num in row]

print(flat)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
['PYTHON', 'IS', 'AWESOME']
[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
['apple', 'banana', 'mango']
['Odd', 'Even', 'Odd', 'Even', 'Odd']
[1, 2, 3, 4, 5, 6]


In [8]:
matrix = [[1, 2], [3, 4], [5, 6]]

for row in matrix:
    for num in row:
        print(num)

# 두 for를 한 줄에 쓰면
# for row in matrix for num in row

# 마지막 num을 앞쪽에
# num for row in matrix for num in row

1
2
3
4
5
6


In [9]:
# 5. dictionary comprehension

square_dict = {x: x**2 for x in range(5)}
print(square_dict)

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}


## 3. pandas MultiIndex 생성하기

### 📚 MultiIndex 생성 방법 용어 정리

| 메서드 | 입력 형태 | 사용 상황 | 결과 |
|--------|----------|-----------|------|
| `from_tuples()` | `[('A',1), ('A',2)]` | 이미 튜플 리스트가 있을 때 | 각 튜플이 하나의 인덱스 |
| `from_product()` | `[['A','B'], [1,2]]` | 모든 조합이 필요할 때 | 데카르트 곱 (A×B) |
| `from_arrays()` | `[['A','A','B','B'], [1,2,1,2]]` | 각 레벨을 별도로 가지고 있을 때 | 병렬 배열 결합 |
| `from_frame()` | DataFrame | DataFrame의 컬럼을 인덱스로 | 각 컬럼이 하나의 레벨 |
| `set_index()` | DataFrame의 컬럼명 | 기존 DataFrame 변환 | 선택한 컬럼이 인덱스로 |

### 데카르트 곱 (Cartesian Product) 이해하기

```
리스트 A: ['California', 'New York', 'Texas']
리스트 B: [2000, 2010]

A × B (데카르트 곱) = 모든 가능한 조합

          2000          2010
           │             │


In [49]:
# pandas MultiIndex 시작

In [10]:
print(index)

[('California', 2000), ('California', 2010), ('New York', 2000), ('New York', 2010), ('Texas', 2000), ('Texas', 2010)]


In [11]:
print(type(index))

<class 'list'>


In [ ]:
# index(list) → index_multi(MultiIndex)

index_multi = pd.MultiIndex.from_tuples(index)
index_multi

MultiIndex([('California', 2000),
            ('California', 2010),
            (  'New York', 2000),
            (  'New York', 2010),
            (     'Texas', 2000),
            (     'Texas', 2010)],
           )

In [13]:
# 1. pd.MultiIndex.from_product: 데카르트 곱으로 MultiIndex 생성
# states와 years의 모든 조합을 생성
# ['California', 'New York', 'Texas'] × [2000, 2010]
# 결과: 3개 주 × 2개 연도 = 6개의 인덱스 조합

states = ['California', 'New York', 'Texas']
years = [2000, 2010]

# names 파라미터로 각 레벨의 이름 지정
index_multi = pd.MultiIndex.from_product([states, years], names=['state', 'year'])
print(index_multi)

MultiIndex([('California', 2000),
            ('California', 2010),
            (  'New York', 2000),
            (  'New York', 2010),
            (     'Texas', 2000),
            (     'Texas', 2010)],
           names=['state', 'year'])


In [14]:
# 2. pd.MultiIndex.from_arrays: 배열 리스트로 MultiIndex 생성
# 각 레벨의 값을 별도의 리스트로 제공
# 첫 번째 리스트: 첫 번째 레벨(state)의 값들
# 두 번째 리스트: 두 번째 레벨(year)의 값들

arrays = [
    ['California', 'California', 'New York', 'New York', 'Texas', 'Texas'],
    [2000, 2010, 2000, 2010, 2000, 2010]
]

index_multi = pd.MultiIndex.from_arrays(arrays, names=['state', 'year'])
print(index_multi)

MultiIndex([('California', 2000),
            ('California', 2010),
            (  'New York', 2000),
            (  'New York', 2010),
            (     'Texas', 2000),
            (     'Texas', 2010)],
           names=['state', 'year'])


In [16]:
# 3. pd.MultiIndex.from_frame: DataFrame의 컬럼을 사용하여 MultiIndex 생성
# DataFrame의 각 컬럼이 MultiIndex의 각 레벨이 됨

df_info = pd.DataFrame({
    'state': ['California', 'California', 'New York', 'New York', 'Texas', 'Texas'],
    'year': [2000, 2010, 2000, 2010, 2000, 2010]
})

index_multi = pd.MultiIndex.from_frame(df_info)
print(index_multi)

MultiIndex([('California', 2000),
            ('California', 2010),
            (  'New York', 2000),
            (  'New York', 2010),
            (     'Texas', 2000),
            (     'Texas', 2010)],
           names=['state', 'year'])


In [17]:
# 4. df.set_index(): 기존 DataFrame에서 MultiIndex 설정
# 여러 컬럼을 리스트로 전달하면 해당 컬럼들이 MultiIndex가 됨

data = {
    'state': ['California', 'California', 'New York', 'New York', 'Texas', 'Texas'],
    'year': [2000, 2010, 2000, 2010, 2000, 2010],
    'pop': [33.8, 37.2, 18.9, 19.3, 20.8, 25.1]
}
df = pd.DataFrame(data)

# 'state'와 'year' 컬럼을 MultiIndex로 설정
df_multi = df.set_index(['state', 'year'])

print(df_multi)

                  pop
state      year      
California 2000  33.8
           2010  37.2
New York   2000  18.9
           2010  19.3
Texas      2000  20.8
           2010  25.1


In [19]:
# 현재 pop
pop

(California, 2000)    33871648
(California, 2010)    37253956
(New York, 2000)      18976457
(New York, 2010)      19378102
(Texas, 2000)         20851820
(Texas, 2010)         25145561
dtype: int64

In [20]:
index_multi

MultiIndex([('California', 2000),
            ('California', 2010),
            (  'New York', 2000),
            (  'New York', 2010),
            (     'Texas', 2000),
            (     'Texas', 2010)],
           names=['state', 'year'])

In [21]:
pop_multi = pop.reindex(index_multi)
pop_multi

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [22]:
type(pop_multi)
# pop_multi의 타입은 Series

pandas.Series

In [ ]:
# MultiIndex를 사용한 효율적인 데이터 접근

# 기존의 비효율적인 방법 (list comprehension 사용):
# pop[[i for i in pop.index if i[1]==2000]]







# MultiIndex를 사용하면 간결하게 접근 가능
# 첫 번째 레벨(:)은 모든 값, 두 번째 레벨(2000)은 2000년만 선택
pop_multi[:, 2000]

# pop_multi의 타입은 Series이기 떄문에 컬럼이 하나밖에 없음,
# pandas는 pop_multi[:, 2000]을 보고 MultiIndex라고 판단

# 주의: pop_multi가 Series일 때만 직접 인덱싱 가능
# DataFrame인 경우 df.loc[], df.iloc[]를 사용해야 함

state
California    33871648
New York      18976457
Texas         20851820
dtype: int64

In [23]:
# 인덱싱 방법 정리를 위한 예제 DataFrame

import pandas as pd

data = {
    'state': ['California', 'New York', 'Texas', 'Florida'],
    'year': [2000, 2010, 2000, 2010],
    'pop': [33.8, 37.2, 20.8, 18.8]
}
df = pd.DataFrame(data)
df

,state,year,pop
0,California,2000,33.8
1,New York,2010,37.2
2,Texas,2000,20.8
3,Florida,2010,18.8


In [ ]:
# 방법 1: df[]를 사용한 컬럼 선택
# 문자열 컬럼명을 사용하면 해당 컬럼(Series) 반환

col_state = df['state']
print(col_state)

# 리스트를 사용하여 여러 컬럼 선택 → DataFrame 반환
# df[['col1', 'col2']]에서 쉼표는 리스트 내부 구분자, DataFrame 인덱싱 문법의 쉼표가 아님
cols = df[['state', 'pop']]
print(cols)


0    California
1      New York
2         Texas
3       Florida
Name: state, dtype: str
        state   pop
0  California  33.8
1    New York  37.2
2       Texas  20.8
3     Florida  18.8


In [25]:
# 방법 2: df[]를 사용한 행 슬라이싱
# 슬라이스 문법(0:2)을 사용하면 행을 선택함
# 주의: 0:2는 0, 1행을 선택 (끝값 2는 포함되지 않음)

rows = df[0:2]
print(rows)

# 중요: df[]에는 쉼표(,)가 없음!
# - 슬라이스 → 행 선택
# - 문자열/리스트 → 컬럼 선택
# df[행, 열] 문법은 없음 → 대신 df.loc[행, 열] 사용


        state  year   pop
0  California  2000  33.8
1    New York  2010  37.2


In [26]:
# 3. 불리언 마스크를 사용하면 행 선택

mask = df['pop'] > 30
print(df[mask])

        state  year   pop
0  California  2000  33.8
1    New York  2010  37.2


In [27]:
# 컬럼 이름을 알려주지 않음

df = pd.DataFrame([[10, 20], [30, 40]])
# pandas가 디폴트 컬럼 이름 0, 1, 2, ...을 사용
# 이 경우 컬럼명은 숫자 타입에 해당
print(df)
print('-'*100)
print(df[0]) 

# print(df['0']) # KeyError: '0'

    0   1
0  10  20
1  30  40
----------------------------------------------------------------------------------------------------
0    10
1    30
Name: 0, dtype: int64


In [74]:
# pd.Series를 데이터프레임에 끼워넣기

pop_multi

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [28]:
pop_df = pd.DataFrame({
    'total': pop_multi, # MultiIndex
    'under18': [9267089, 9284094, 4687374, 4318033, 5906301, 6879014],
})

pop_df

total  under18
state      year                   
California 2000  33871648  9267089
           2010  37253956  9284094
New York   2000  18976457  4687374
           2010  19378102  4318033
Texas      2000  20851820  5906301
           2010  25145561  6879014

In [29]:
# 인덱스 이름 - 다 만들고 나중에 이름 붙이기

rng = np.random.RandomState(42)
rng.rand(6, 3)

array([[0.37454012, 0.95071431, 0.73199394],
       [0.59865848, 0.15601864, 0.15599452],
       [0.05808361, 0.86617615, 0.60111501],
       [0.70807258, 0.02058449, 0.96990985],
       [0.83244264, 0.21233911, 0.18182497],
       [0.18340451, 0.30424224, 0.52475643]])

In [32]:
# rng.rand(6, 3)는 난수 생성기(rng)를 사용해서 6행 3열의 2차원 배열을 생성
df = pd.DataFrame(rng.rand(6, 3), 
                  index=[['A', 'A', 'B', 'B', 'C', 'C'], [1,2,1,2,1,2]],
                  columns= ['col1', 'col2', 'col3']
                  )

df

col1      col2      col3
A 1  0.597900  0.921874  0.088493
  2  0.195983  0.045227  0.325330
B 1  0.388677  0.271349  0.828738
  2  0.356753  0.280935  0.542696
C 1  0.140924  0.802197  0.074551
  2  0.986887  0.772245  0.198716

In [33]:
df.index.names = ['index1', 'index2']
df

col1      col2      col3
index1 index2                              
A      1       0.597900  0.921874  0.088493
       2       0.195983  0.045227  0.325330
B      1       0.388677  0.271349  0.828738
       2       0.356753  0.280935  0.542696
C      1       0.140924  0.802197  0.074551
       2       0.986887  0.772245  0.198716

In [35]:
# 인덱스 이름 - 만들면서 알려주기

index_arr = [['A', 'A', 'B', 'B', 'C', 'C'], [1,2,1,2,1,2]]
index = pd.MultiIndex.from_arrays(index_arr, names= ('index1', 'index2'))
#                                            ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑

df = pd.DataFrame(rng.rand(6,3), 
                  index=index,
                  columns= ['col1', 'col2', 'col3']
                  )

df

col1      col2      col3
index1 index2                              
A      1       0.119594  0.713245  0.760785
       2       0.561277  0.770967  0.493796
B      1       0.522733  0.427541  0.025419
       2       0.107891  0.031429  0.636410
C      1       0.314356  0.508571  0.907566
       2       0.249292  0.410383  0.755551

## 5. 컬럼에도 MultiIndex 적용하기

MultiIndex는 행 인덱스뿐만 아니라 컬럼에도 적용할 수 있습니다.

### 예시: 건강 데이터
- **행 MultiIndex**: Year(연도) × Visit(방문 차수)
- **컬럼 MultiIndex**: Patient(환자) × Data type(데이터 종류: HR, Temp)

이렇게 하면 4차원 데이터(Year × Visit × Patient × Data type)를 2D 테이블로 표현 가능

In [ ]:
# 행과 컬럼 모두에 MultiIndex 생성
row_index = pd.MultiIndex.from_product(
    [[2013, 2014], ["1st", "2nd"]], names=["Year", "Visit"]
)
col_index = pd.MultiIndex.from_product(
    [["Bob", "Guido", "Sue"], ["HR", "Temp"]], names=["Patient", "Data type"]
)
print(row_index)
print(col_index)

MultiIndex([(2013, '1st'),
            (2013, '2nd'),
            (2014, '1st'),
            (2014, '2nd')],
           names=['Year', 'Visit'])
MultiIndex([(  'Bob',   'HR'),
            (  'Bob', 'Temp'),
            ('Guido',   'HR'),
            ('Guido', 'Temp'),
            (  'Sue',   'HR'),
            (  'Sue', 'Temp')],
           names=['Patient', 'Data type'])


In [37]:
# 샘플 건강 데이터 생성
# HR(심박수): 50~79 사이의 정수 12개
# Temp(체온): 평균 37도, 표준편차 1인 정규분포에서 12개 샘플

data_hr = np.random.randint(50, 80, 12)
data_temp = 37 + np.random.randn(12)

# zip으로 HR과 Temp를 쌍으로 묶어서 리스트 생성
data_x = [i for i in zip(data_hr, data_temp)]
data_x  

[(np.int32(54), np.float64(37.615412122590634)),
 (np.int32(77), np.float64(36.82264728169202)),
 (np.int32(60), np.float64(35.87811038831361)),
 (np.int32(75), np.float64(36.17121852034546)),
 (np.int32(78), np.float64(36.90808804667133)),
 (np.int32(61), np.float64(37.20315495690046)),
 (np.int32(79), np.float64(35.682056098003336)),
 (np.int32(58), np.float64(37.44020260180181)),
 (np.int32(76), np.float64(36.79320348783259)),
 (np.int32(53), np.float64(37.78985425974147)),
 (np.int32(63), np.float64(38.3077040146596)),
 (np.int32(52), np.float64(38.27825219252953))]

In [38]:
arr = np.array(data_x)
arr

array([[54.        , 37.61541212],
       [77.        , 36.82264728],
       [60.        , 35.87811039],
       [75.        , 36.17121852],
       [78.        , 36.90808805],
       [61.        , 37.20315496],
       [79.        , 35.6820561 ],
       [58.        , 37.4402026 ],
       [76.        , 36.79320349],
       [53.        , 37.78985426],
       [63.        , 38.30770401],
       [52.        , 38.27825219]])

In [39]:
print(arr.shape)
print(arr)

(12, 2)
[[54.         37.61541212]
 [77.         36.82264728]
 [60.         35.87811039]
 [75.         36.17121852]
 [78.         36.90808805]
 [61.         37.20315496]
 [79.         35.6820561 ]
 [58.         37.4402026 ]
 [76.         36.79320349]
 [53.         37.78985426]
 [63.         38.30770401]
 [52.         38.27825219]]


In [41]:
# Reshape to 4 rows and 6 cols
arr_multi = arr.reshape(4,6)
arr_multi

array([[54.        , 37.61541212, 77.        , 36.82264728, 60.        ,
        35.87811039],
       [75.        , 36.17121852, 78.        , 36.90808805, 61.        ,
        37.20315496],
       [79.        , 35.6820561 , 58.        , 37.4402026 , 76.        ,
        36.79320349],
       [53.        , 37.78985426, 63.        , 38.30770401, 52.        ,
        38.27825219]])

In [42]:
health_data = pd.DataFrame(arr_multi, columns=col_index, index=row_index)
health_data

Patient      Bob            Guido              Sue           
Data type     HR       Temp    HR       Temp    HR       Temp
Year Visit                                                   
2013 1st    54.0  37.615412  77.0  36.822647  60.0  35.878110
     2nd    75.0  36.171219  78.0  36.908088  61.0  37.203155
2014 1st    79.0  35.682056  58.0  37.440203  76.0  36.793203
     2nd    53.0  37.789854  63.0  38.307704  52.0  38.278252

In [43]:
health_data['Bob']

Data type     HR       Temp
Year Visit                 
2013 1st    54.0  37.615412
     2nd    75.0  36.171219
2014 1st    79.0  35.682056
     2nd    53.0  37.789854

In [44]:
health_data['Bob']['HR']

Year  Visit
2013  1st      54.0
      2nd      75.0
2014  1st      79.0
      2nd      53.0
Name: HR, dtype: float64

In [45]:
# MultiIndex DataFrame에서 데이터 접근
# loc을 사용하여 특정 행과 컬럼 선택

# 2014년도 Bob의 HR 데이터 선택
# 주의: 튜플 ('Bob','HR')은 계층을 표현 (첫 번째 레벨 'Bob', 두 번째 레벨 'HR')
# 리스트 ['Bob','HR']은 단순 나열로 해석되어 오류 발생

health_data.loc[2014, ('Bob','HR')]

Visit
1st    79.0
2nd    53.0
Name: (Bob, HR), dtype: float64

---

## MultiIndex 요약

### 1. MultiIndex 생성 방법

| 메서드 | 설명 |
|--------|------|
| `from_tuples()` | 튜플 리스트로 생성 |
| `from_product()` | 데카르트 곱으로 생성 |
| `from_arrays()` | 배열 리스트로 생성 |
| `from_frame()` | DataFrame으로 생성 |
| `set_index()` | 기존 DataFrame에서 설정 |

### 2. MultiIndex 인덱싱

```python
# Series (MultiIndex)
series[:, 2000]  # 두 번째 레벨이 2000인 모든 값

# DataFrame (MultiIndex)
df.loc['A', :]        # 첫 번째 레벨이 'A'인 모든 행
df.loc[:, ('A','B')]  # 컬럼 MultiIndex에서 ('A','B') 컬럼 선택
```

### 3. 주의사항
- 튜플 `('A','B')`: 계층 표현 (MultiIndex 레벨 지정)
- 리스트 `['A','B']`: 단순 나열 (여러 개별 항목)
- DataFrame은 `df[]`에 쉼표 사용 불가 → `df.loc[]` 사용